# Mini reto · Deja tu cuaderno reproducible
**Curso de Capacitación en Infraestructura Computacional para Finanzas Cuantitativas · Tema 2: Ambientes de desarrollo y gestión de entornos**

**Tu misión (9 minutos):**
1. Ejecuta la celda **Entorno** y mira tus versiones.
2. Genera tu `requirements.txt`.
3. Reinicia la sesión (así simulas a un colega que abre tu cuaderno) y reinstala desde la lista.
4. Ejecuta el modelo y verifica que tu **AUC** coincide con el del docente.

**Antes de empezar:** copia este cuaderno a tu Drive (*Archivo → Guardar una copia en Drive*) y ejecuta las celdas **en orden**.

## Celda 1 · Entorno
Muestra la versión de Python y de las librerías que usa el modelo.

In [ ]:
import sys, importlib.metadata as md
print("Python", sys.version.split()[0])
for lib in ["numpy", "pandas", "scikit-learn"]:
    print(f"{lib}=={md.version(lib)}")

## Celda 2 · Generar `requirements.txt`
Escribe la lista de compras de tu proyecto: una librería por línea, con su versión exacta.

In [ ]:
from pathlib import Path

libs = ["numpy", "pandas", "scikit-learn"]
lineas = [f"{l}=={md.version(l)}" for l in libs]
Path("requirements.txt").write_text("\n".join(lineas))
print(Path("requirements.txt").read_text())

> **Cuidado:** los archivos de Colab son temporales. Si "eliminas el entorno de ejecución", se borran. Para conservarlo, ejecuta la celda opcional del final (descargar) o guárdalo en Drive.

## Paso intermedio · Simula a un colega
Menú **Entorno de ejecución → Reiniciar sesión**. Esto borra lo que está en memoria pero **conserva tus archivos**. Después de reiniciar, ejecuta desde la Celda 3 (la Celda 1 ya no hace falta).

## Celda 3 · Reinstalar desde la lista
Instala exactamente lo que dice `requirements.txt`. Como son las mismas versiones que ya tienes, será casi inmediato.

In [ ]:
!pip install -q -r requirements.txt
print("Listo: entorno reconstruido desde requirements.txt")

## Celda 4 · El modelo
Un modelo sencillo de probabilidad de default con datos inventados y una **semilla fija** (42): el resultado debe ser el mismo para todos.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def generar(n, rng):
    return pd.DataFrame({
        "ingreso": rng.normal(3500, 1200, n).clip(900),
        "deuda_ingreso": rng.beta(2, 5, n),
        "antiguedad": rng.integers(0, 20, n),
        "atrasos_prev": rng.poisson(0.6, n)})

def etiquetar(df, rng):
    z = (-3.2 + 4.0*df.deuda_ingreso
         + 0.45*df.atrasos_prev - 0.0003*df.ingreso
         - 0.03*df.antiguedad)
    df["default"] = rng.binomial(1, 1/(1 + np.exp(-z)))
    return df

rng = np.random.default_rng(42)
df = etiquetar(generar(20000, rng), rng)

FEATURES = ["ingreso", "deuda_ingreso", "antiguedad", "atrasos_prev"]
X, y = df[FEATURES], df["default"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(Xtr, ytr)
auc = roc_auc_score(yte, modelo.predict_proba(Xte)[:, 1])
print("AUC:", round(auc, 3))

### ¿Coincide?
El AUC esperado es cercano a **0.708**. Si el tuyo es distinto: revisa que no cambiaste la semilla y que ejecutaste todas las celdas en orden.

**Para conversar:** ¿qué pasaría si tu colega tuviera otras versiones de las librerías? ¿Qué parte del entorno **no** quedó registrada en `requirements.txt`? (pista: la versión de Python y los datos)

## Opcional · Conservar tu `requirements.txt`

In [ ]:
from google.colab import files
files.download("requirements.txt")